# Browser

> Main file browser component that composes all sub-components.

In [ ]:
#| default_exp components.browser

In [ ]:
#| export
from typing import Any, Optional

from fasthtml.common import Div

# DaisyUI components
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, border_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_direction, grow
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.core.base import combine_classes

# Local imports
from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.core.models import DirectoryListing, BrowserState
from cjm_fasthtml_file_browser.core.html_ids import FileBrowserHtmlIds
from cjm_fasthtml_file_browser.components.path_bar import render_path_bar
from cjm_fasthtml_file_browser.components.toolbar import render_toolbar
from cjm_fasthtml_file_browser.components.listing import render_listing

## Main Browser Component

The primary public API for rendering the file browser.

In [ ]:
#| export
def render_file_browser(
    listing: DirectoryListing,              # Current directory listing
    config: FileBrowserConfig,              # Browser configuration
    state: BrowserState,                    # Current browser state
    navigate_url: str,                      # URL for directory navigation
    select_url: str,                        # URL for file selection
    toggle_view_url: str,                   # URL for view mode toggle
    change_sort_url: str,                   # URL for sort change
    refresh_url: str,                       # URL for refresh
    path_input_url: str = "",               # URL for path input (optional)
    home_path: str = "",                    # Home directory path
    hx_target: Optional[str] = None,        # Override HTMX target (default: container_id)
    select_hx_target: Optional[str] = None, # Override select target (default: content_id)
    select_hx_swap: Optional[str] = None,   # Override select swap (default: "outerHTML")
) -> Any:  # Complete file browser component
    """
    Render the complete file browser component.
    
    Select operations target the content wrapper (not the full container) so that
    path bar and toolbar are preserved. Consuming apps should include a scroll
    preservation script for the content wrapper to maintain scroll position on select.
    
    Component hierarchy:
    ```
    render_file_browser(listing, config, state, ...)
    ├── render_path_bar(...)        [if config.show_path_bar]
    │   ├── nav_buttons()
    │   └── breadcrumbs() OR path_input()
    ├── render_toolbar(...)         [if config.show_toolbar]
    │   ├── view_mode_toggle()
    │   └── sort_controls()
    └── render_listing(...)
        ├── parent_navigation()
        └── items[]
            └── render_item() [file or folder]
    ```
    """
    ids = FileBrowserHtmlIds()
    # Use provided hx_target or default to container_id
    target = hx_target or FileBrowserHtmlIds.as_selector(config.container_id)
    # Select operations target the content wrapper (preserves path bar + toolbar)
    _select_target = select_hx_target or FileBrowserHtmlIds.as_selector(config.content_id)
    _select_swap = select_hx_swap or "outerHTML"
    
    children = []
    
    # Path bar
    if config.show_path_bar:
        children.append(render_path_bar(
            current_path=listing.path,
            parent_path=listing.parent_path,
            home_path=home_path or state.current_path,
            config=config,
            navigate_url=navigate_url,
            refresh_url=refresh_url,
            hx_target=target,
            path_bar_id=ids.PATH_BAR,
        ))
    
    # Toolbar
    if config.show_toolbar:
        children.append(render_toolbar(
            state=state,
            config=config,
            toggle_url=toggle_view_url,
            sort_url=change_sort_url,
            hx_target=target,
            toolbar_id=ids.TOOLBAR,
        ))
    
    # Directory listing
    listing_content = render_listing(
        listing=listing,
        config=config,
        state=state,
        navigate_url=navigate_url,
        select_url=select_url,
        hx_target=target,
        listing_id=ids.LISTING,
        select_hx_target=_select_target,
        select_hx_swap=_select_swap,
    )
    
    # Wrap listing in scrollable container (use config.content_id)
    children.append(Div(
        listing_content,
        id=config.content_id,
        cls=combine_classes(grow(), overflow.auto, min_h(48))
    ))
    
    # Build main container
    return Div(
        *children,
        id=config.container_id,
        cls=combine_classes(
            flex_display, flex_direction.col,
            w.full, h.full,
            border(), border_dui.base_300, border_radius.box,
            bg_dui.base_100,
            overflow.hidden
        )
    )

In [ ]:
from fasthtml.common import to_xml
from cjm_file_discovery.core.models import FileInfo

# Test complete browser
config = FileBrowserConfig()
state = BrowserState(current_path="/home/user", view_mode="list")

listing = DirectoryListing(
    path="/home/user",
    items=[
        FileInfo(name="Documents", path="/home/user/Documents", is_directory=True),
        FileInfo(name="file.txt", path="/home/user/file.txt", is_directory=False, size=1024),
    ],
    parent_path="/home"
)

browser = render_file_browser(
    listing=listing,
    config=config,
    state=state,
    navigate_url="/browser/navigate",
    select_url="/browser/select",
    toggle_view_url="/browser/toggle_view",
    change_sort_url="/browser/change_sort",
    refresh_url="/browser/refresh",
)
html = to_xml(browser)

# Verify structure
assert "file-browser" in html  # Container ID
assert "breadcrumbs" in html   # Path bar with breadcrumbs
assert "Documents" in html     # Folder item
assert "file.txt" in html      # File item
assert "<table" in html.lower()  # List view
assert "<button" in html.lower()  # Toolbar buttons
assert "<select" in html.lower()  # Sort dropdown

print("Complete browser tests passed!")

Complete browser tests passed!


In [ ]:
# Test browser with different configurations

# No path bar
config_no_path = FileBrowserConfig(show_path_bar=False)
browser = render_file_browser(
    listing=listing,
    config=config_no_path,
    state=state,
    navigate_url="/browser/navigate",
    select_url="/browser/select",
    toggle_view_url="/browser/toggle_view",
    change_sort_url="/browser/change_sort",
    refresh_url="/browser/refresh",
)
html = to_xml(browser)
assert "breadcrumbs" not in html

# No toolbar
config_no_toolbar = FileBrowserConfig(show_toolbar=False)
browser = render_file_browser(
    listing=listing,
    config=config_no_toolbar,
    state=state,
    navigate_url="/browser/navigate",
    select_url="/browser/select",
    toggle_view_url="/browser/toggle_view",
    change_sort_url="/browser/change_sort",
    refresh_url="/browser/refresh",
)
html = to_xml(browser)
assert "<select" not in html.lower()  # No sort dropdown

# Grid view
state_grid = BrowserState(current_path="/home/user", view_mode="grid")
browser = render_file_browser(
    listing=listing,
    config=config,
    state=state_grid,
    navigate_url="/browser/navigate",
    select_url="/browser/select",
    toggle_view_url="/browser/toggle_view",
    change_sort_url="/browser/change_sort",
    refresh_url="/browser/refresh",
)
html = to_xml(browser)
assert "grid" in html  # Grid layout

print("Configuration variants tests passed!")

Configuration variants tests passed!


## Content-Only Render

For select-triggered HTMX updates, render just the content wrapper with listing.

## Scroll Preservation Script

Generates a JS snippet that saves and restores scroll position on the content wrapper
when it is replaced via `outerHTML` during select operations.

In [ ]:
#| export
def render_browser_content(
    listing: DirectoryListing,              # Current directory listing
    config: FileBrowserConfig,              # Browser configuration
    state: BrowserState,                    # Current browser state
    navigate_url: str,                      # URL for directory navigation
    select_url: str,                        # URL for file selection
    hx_target: Optional[str] = None,        # Override HTMX target (default: container_id)
    select_hx_target: Optional[str] = None, # Override select target (default: content_id)
    select_hx_swap: Optional[str] = None,   # Override select swap (default: "outerHTML")
) -> Any:  # Content wrapper with listing (for outerHTML swap on content_id)
    """Render the content wrapper with listing for select-triggered HTMX updates.
    
    Returns the scrollable content wrapper div (with its ID) containing the listing.
    Used as the outerHTML replacement for the content wrapper element, preserving
    path bar and toolbar while replacing only the scrollable listing area.
    """
    ids = FileBrowserHtmlIds()
    target = hx_target or FileBrowserHtmlIds.as_selector(config.container_id)
    _select_target = select_hx_target or FileBrowserHtmlIds.as_selector(config.content_id)
    _select_swap = select_hx_swap or "outerHTML"
    
    listing_content = render_listing(
        listing=listing,
        config=config,
        state=state,
        navigate_url=navigate_url,
        select_url=select_url,
        hx_target=target,
        listing_id=ids.LISTING,
        select_hx_target=_select_target,
        select_hx_swap=_select_swap,
    )
    
    return Div(
        listing_content,
        id=config.content_id,
        cls=combine_classes(grow(), overflow.auto, min_h(48))
    )

In [ ]:
#| export
def generate_scroll_preservation_script(
    content_id: str,  # HTML ID of the scrollable content wrapper
) -> str:  # JavaScript snippet for scroll preservation
    """Generate a JS snippet that preserves scroll position on outerHTML swaps.
    
    Include this script on the page (e.g., via `Script(generate_scroll_preservation_script(...))`).
    It listens for HTMX swap events targeting the content wrapper and saves/restores scrollTop.
    """
    return f"""(function() {{
    var saved = null;
    document.addEventListener('htmx:beforeSwap', function(e) {{
        if (e.detail.target && e.detail.target.id === '{content_id}') {{
            saved = e.detail.target.scrollTop;
        }}
    }});
    document.addEventListener('htmx:afterSwap', function(e) {{
        if (saved != null) {{
            var el = document.getElementById('{content_id}');
            if (el) {{ el.scrollTop = saved; saved = null; }}
        }}
    }});
}})();"""

In [ ]:
# Test content-only render
content = render_browser_content(
    listing=listing,
    config=config,
    state=state,
    navigate_url="/browser/navigate",
    select_url="/browser/select",
)
html = to_xml(content)
assert "Documents" in html
assert "file.txt" in html
# Should not have path bar or toolbar
assert "breadcrumbs" not in html

print("Content-only render tests passed!")

Content-only render tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()